<a href="https://colab.research.google.com/github/youssef-bg-15/Freecel/blob/main/Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 13.9 MB/s eta 0:00:00


In [ ]:
!pip install evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 2.7 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM, TrainingArguments, Trainer
from datasets import load_dataset
from evaluate import load
import torch

In [ ]:
# Chargement du dataset DialogSum
df = load_dataset("knkarthick/dialogsum")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/4.65k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/11.3M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/442k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12460 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [ ]:
df

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

In [ ]:
print("dialogue : " ,df['train'][1]['dialogue'])
print("summary :" ,df['train'][1]['summary'])

dialogue :  #Person1#: Hello Mrs. Parker, how have you been?
#Person2#: Hello Dr. Peters. Just fine thank you. Ricky and I are here for his vaccines.
#Person1#: Very well. Let's see, according to his vaccination record, Ricky has received his Polio, Tetanus and Hepatitis B shots. He is 14 months old, so he is due for Hepatitis A, Chickenpox and Measles shots.
#Person2#: What about Rubella and Mumps?
#Person1#: Well, I can only give him these for now, and after a couple of weeks I can administer the rest.
#Person2#: OK, great. Doctor, I think I also may need a Tetanus booster. Last time I got it was maybe fifteen years ago!
#Person1#: We will check our records and I'll have the nurse administer and the booster as well. Now, please hold Ricky's arm tight, this may sting a little.
summary : Mrs Parker takes Ricky for his vaccines. Dr. Peters checks the record and then gives Ricky a vaccine.


In [ ]:
# Utilisation d'un modèle pré-entraîné pour le résumé
text_summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
article_1 = df['train'][1]['dialogue']
print("Résumé initial:", text_summarizer(article_1, max_length=20, min_length=10, do_sample=False))


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cuda:0


Résumé initial: [{'summary_text': 'Ricky has received his Polio, Tetanus and Hepatitis B shots.'}]


In [ ]:
# Fine-tuning du modèle
# Choix du modèle BART (Encoder-Decoder)
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-base")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-base")

config.json:   0%|          | 0.00/1.72k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

In [ ]:
# Prétraitement des données
def preprocess_function(batch):
    source = batch['dialogue']
    target = batch["summary"]
    source_ids = tokenizer(source, truncation=True, padding="max_length", max_length=128)
    target_ids = tokenizer(target, truncation=True, padding="max_length", max_length=128)

    labels = target_ids["input_ids"]
    labels = [[(label if label != tokenizer.pad_token_id else -100) for label in labels_example] for labels_example in labels]

    return {
        "input_ids": source_ids["input_ids"],
        "attention_mask": source_ids["attention_mask"],
        "labels": labels
    }

In [ ]:
df_source = df.map(preprocess_function, batched=True)


Map:   0%|          | 0/12460 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [ ]:
# Configuration des hyperparamètres
training_args = TrainingArguments(
    output_dir="/content",  # Répertoire de sauvegarde
    per_device_train_batch_size=8,
    num_train_epochs=3,  # Augmenté pour une meilleure convergence
    learning_rate=5e-5,  # Apprentissage adapté pour stabiliser l'entraînement
    weight_decay=0.01,
    warmup_steps=500,  # Scheduler de taux d'apprentissage
    logging_dir="./logs",
    remove_unused_columns=False
)

In [ ]:
# Création de l'entraîneur
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=df_source["train"],
    eval_dataset=df_source["test"]
)

In [ ]:
# Entraînement du modèle
trainer.train()

Step,Training Loss
500,1.297500
1000,1.302200
1500,1.318100
2000,1.165200
2500,1.154100
3000,1.167300
3500,1.040400
4000,1.004200
4500,1.008100


TrainOutput(global_step=4674, training_loss=1.1568962361601411, metrics={'train_runtime': 1872.988, 'train_samples_per_second': 19.957, 'train_steps_per_second': 2.495, 'total_flos': 2866066221957120.0, 'train_loss': 1.1568962361601411, 'epoch': 3.0})

In [ ]:
eval_results = trainer.evaluate()
print("Résultats d'évaluation:", eval_results)

Résultats d'évaluation: {'eval_loss': 1.7762255668640137, 'eval_runtime': 19.3432, 'eval_samples_per_second': 77.547, 'eval_steps_per_second': 9.719, 'epoch': 3.0}


In [ ]:
# Sauvegarde du modèle et du tokenizer
model.save_pretrained("/content/my_new_model")
tokenizer.save_pretrained("/content/my_new_model")

('/content/my_new_model/tokenizer_config.json',
 '/content/my_new_model/special_tokens_map.json',
 '/content/my_new_model/vocab.json',
 '/content/my_new_model/merges.txt',
 '/content/my_new_model/added_tokens.json',
 '/content/my_new_model/tokenizer.json')

In [ ]:
# Détecter le device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Charger le modèle sur le bon device
model.to(device)

# Modifier la fonction de résumé
def summarize(blog_post):
    # Tokenize le texte et place les tensors sur le bon device
    inputs = tokenizer(blog_post, max_length=1024, truncation=True, return_tensors="pt").to(device)

    # Générer le résumé
    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=150,
        min_length=40,
        length_penalty=2.0,
        num_beams=4,
        early_stopping=True
    )

    # Décoder le résumé
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

In [ ]:
# Libérer la mémoire GPU
del model
del trainer
torch.cuda.empty_cache()

In [ ]:
# Sauvegarde
tokenizer.save_pretrained("/content/my_new_model")
model = AutoModelForSeq2SeqLM.from_pretrained("/content/my_new_model")  # Recharge temporairement pour sauvegarde
model.save_pretrained("/content/my_new_model")

In [ ]:
# Comparaison ROUGE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
tokenizer_pretrained = AutoTokenizer.from_pretrained("facebook/bart-base")
model_pretrained = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-base")
model_pretrained.to(device)

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50265, 768, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50265, 768, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
      (layers): ModuleList(
        (0-5): 6 x BartEncoderLayer(
          (self_attn): BartSdpaAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (final_lay

In [ ]:
tokenizer_finetuned = AutoTokenizer.from_pretrained("/content/my_new_model")
model_finetuned = AutoModelForSeq2SeqLM.from_pretrained("/content/my_new_model")
model_finetuned.to(device)

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50265, 768, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50265, 768, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
      (layers): ModuleList(
        (0-5): 6 x BartEncoderLayer(
          (self_attn): BartSdpaAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (final_lay

In [ ]:
def summarize_pretrained(text):
    inputs = tokenizer_pretrained(text, max_length=1024, truncation=True, return_tensors="pt").to(device)
    summary_ids = model_pretrained.generate(inputs["input_ids"], max_length=150, min_length=40, length_penalty=2.0, num_beams=4, early_stopping=True)
    return tokenizer_pretrained.decode(summary_ids[0], skip_special_tokens=True)

def summarize_finetuned(text):
    inputs = tokenizer_finetuned(text, max_length=1024, truncation=True, return_tensors="pt").to(device)
    summary_ids = model_finetuned.generate(inputs["input_ids"], max_length=150, min_length=40, length_penalty=2.0, num_beams=4, early_stopping=True)
    return tokenizer_finetuned.decode(summary_ids[0], skip_special_tokens=True)

In [ ]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24935 sha256=c2f53a6643f54f5b861999db73e0a65113a5d53ca67a501a618cfb6978de1a43
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge_score


In [ ]:
metric = load("rouge")
test_dataset = df["test"].select(range(100))
dialogues = [example["dialogue"] for example in test_dataset]
references = [example["summary"] for example in test_dataset]

In [ ]:
pretrained_summaries = [summarize_pretrained(dialogue) for dialogue in dialogues]
finetuned_summaries = [summarize_finetuned(dialogue) for dialogue in dialogues]

In [ ]:
pretrained_results = metric.compute(predictions=pretrained_summaries, references=references)
finetuned_results = metric.compute(predictions=finetuned_summaries, references=references)

In [ ]:
print("\n=== Comparaison ROUGE ===")
print("Modèle pré-entraîné (bart-base):", pretrained_results)
print("Modèle fine-tuné:", finetuned_results)


=== Comparaison ROUGE ===
Modèle pré-entraîné (bart-base): {'rouge1': 0.1613702228216507, 'rouge2': 0.03371173038455281, 'rougeL': 0.11838070960436534, 'rougeLsum': 0.11834258341615425}
Modèle fine-tuné: {'rouge1': 0.32890023257985257, 'rouge2': 0.09760316074618718, 'rougeL': 0.2516226222501438, 'rougeLsum': 0.2511514777368922}


In [ ]:
for i in range(3):
    print(f"\nDialogue {i+1}:", dialogues[i])
    print("Référence:", references[i])
    print("Pré-entraîné:", pretrained_summaries[i])
    print("Fine-tuné:", finetuned_summaries[i])


Dialogue 1: #Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to all employees by this afternoon. Are you ready?
#Person2#: Yes, sir. Go ahead.
#Person1#: Attention all staff... Effective immediately, all office communications are restricted to email correspondence and official memos. The use of Instant Message programs by employees during working hours is strictly prohibited.
#Person2#: Sir, does this apply to intra-office communications only? Or will it also restrict external communications?
#Person1#: It should apply to all communications, not only in this office between employees, but also any outside communications.
#Person2#: But sir, many employees use Instant Messaging to communicate with their clients.
#Person1#: They will just have to change their communication methods. I don't want any - one using Instant Messaging in this office. It wastes too much time! Now, please continue with